# FlyRank Search Intelligence — Ranking Lifecycle

**Research question:** Can observable content and search-context signals be used to rank
content observations by relative search-performance opportunity, and does that ranking
generalize to unseen clients?

**Decision supported:** Which content observations should be prioritized for further
review based on their observed search-performance profile?

This notebook is organized to mirror the research paper: Question → Data → Methodology →
Results (vs. baseline) → Limitations → Ranked recommendations → Paper artifacts →
Self-check. All claims in this notebook are **observational and decision-support
oriented** — nothing here claims to predict Google's ranking algorithm, prove causality,
or guarantee that any action will improve rankings.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrimJain-quantum/internship-remote/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Primary research question**

> Can observable content and search-context signals be used to rank content observations
> by relative search-performance opportunity, and does that ranking generalize to unseen
> clients?

**Decision this supports**

> Which content observations should be prioritized for further review based on their
> observed search-performance profile?

**What this system is NOT**

This is an *opportunity-prioritization* ranking, not a causal model. It does not claim
that any feature causes Google rankings, that the model predicts Google's algorithm, that
refreshing content will improve rankings, or that the model predicts future ranking
changes. Findings are described with words like *observed*, *associated with*,
*directional*, and *review candidate* rather than causal language.


In [1]:
# Standard imports used throughout the notebook.
import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ARTIFACT_DIR = Path("artifacts")
CHART_DIR = Path("charts")
ARTIFACT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)

print("Environment ready. Random seed fixed at", RANDOM_SEED)


Environment ready. Random seed fixed at 42


## 2. Data

Source: pseudonymized FlyRank ML Internship dataset, ~205,749 observations × 34 columns,
provided as a Parquet file. No client names, domains, URLs, private queries, or
credentials are exposed anywhere in this notebook's outputs or artifacts.

### 2.1 Load


In [2]:
from pathlib import Path
import pandas as pd


def resolve_data_path() -> str:
    """
    Resolve the ranking_lifecycle dataset.

    Primary source:
        Hugging Face Storage Bucket

    The pipeline uses the Hugging Face bucket directly rather than
    relying on Kaggle-specific filesystem paths.
    """

    hf_path = (
        "hf://buckets/"
        "agrimjain015/internship-lanes-bucket/"
        "default_lanes/ranking_lifecycle.parquet"
    )

    return hf_path


DATA_PATH = resolve_data_path()

print("Using data source:")
print(DATA_PATH)

try:
    df_raw = pd.read_parquet(DATA_PATH)

except Exception as e:
    raise RuntimeError(
        "Could not load ranking_lifecycle.parquet from the "
        "Hugging Face Storage Bucket.\n"
        f"Data source: {DATA_PATH}\n"
        f"Original error: {e}"
    ) from e

print("\nShape:", df_raw.shape)

Using data source:
hf://buckets/agrimjain015/internship-lanes-bucket/default_lanes/ranking_lifecycle.parquet

Shape: (205749, 34)


### 2.2 Required-column and schema check

In [3]:
REQUIRED_COLUMNS = [
    "client_hash_id", "content_hash_id", "keyword_hash_id",
    "keyword_char_count", "keyword_token_count", "public_url_hash_id",
    "public_url_char_count", "public_url_path_depth", "content_title_hash_id",
    "content_title_char_count", "content_title_token_count", "search_volume",
    "content_type", "main_intent", "content_created_at", "content_age_days",
    "client_has_gsc", "client_has_ga4", "impressions_30d", "clicks_30d",
    "pageviews_30d", "sessions_30d", "users_30d", "engaged_sessions_30d",
    "ai_sessions_30d", "scroll_events_30d", "ctr_30d", "avg_position_30d",
    "engagement_rate_30d", "scroll_rate_30d", "ai_traffic_pct_30d",
    "impression_tier", "position_tier", "health_score",
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"
print("All", len(REQUIRED_COLUMNS), "required columns present.")


All 34 required columns present.


### 2.3 Data audit — shape, dtypes, duplicates, missingness, uniqueness

In [4]:
def audit_data(df: pd.DataFrame) -> dict:
    missing = df.isna().sum()
    return {
        "shape": df.shape,
        "dtypes": df.dtypes.astype(str).to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_counts": missing[missing > 0].sort_values(ascending=False).to_dict(),
        "missing_pct": (missing[missing > 0] / len(df) * 100).round(2)
                         .sort_values(ascending=False).to_dict(),
        "n_unique_clients": df["client_hash_id"].nunique(),
        "n_unique_content": df["content_hash_id"].nunique(),
    }

audit = audit_data(df_raw)
print("Shape:", audit["shape"])
print("Duplicate rows:", audit["duplicate_rows"])
print("Unique clients:", audit["n_unique_clients"])
print("Unique content observations:", audit["n_unique_content"])
print()
missing_df = pd.DataFrame({
    "missing_count": audit["missing_counts"],
    "missing_pct": audit["missing_pct"],
})
missing_df.index.name = "column"
missing_df


Shape: (205749, 34)
Duplicate rows: 0
Unique clients: 54
Unique content observations: 205749



,missing_count,missing_pct
column,,
engagement_rate_30d,102382,49.76
ai_traffic_pct_30d,102382,49.76
scroll_rate_30d,102380,49.76
sessions_30d,53587,26.04
pageviews_30d,53587,26.04
ai_sessions_30d,53587,26.04
users_30d,53587,26.04
engaged_sessions_30d,53587,26.04
scroll_events_30d,53587,26.04


### 2.4 Date parsing

`content_created_at` is stored as a JSON-like string, e.g. `{"value":"2025-12-11T09:37:35.216Z"}`.
It is parsed defensively (unparsable values become `NaT` and are counted, not silently
dropped). This dataset is a **cross-sectional snapshot** — the parsed date is used only
for descriptive reporting (e.g. `content_age_days` context) and is explicitly **not**
used to build a chronological split.


In [5]:
def parse_content_created_at(df: pd.DataFrame) -> pd.DataFrame:
    def _parse(x):
        if pd.isna(x):
            return pd.NaT
        try:
            obj = json.loads(x)
            return pd.to_datetime(obj.get("value"), utc=True, errors="coerce")
        except (json.JSONDecodeError, TypeError, AttributeError):
            return pd.to_datetime(x, utc=True, errors="coerce")
    df = df.copy()
    df["content_created_at_parsed"] = df["content_created_at"].apply(_parse)
    return df

df_raw = parse_content_created_at(df_raw)
n_unparsable = df_raw["content_created_at_parsed"].isna().sum()
print("Date range:", df_raw["content_created_at_parsed"].min(), "to",
      df_raw["content_created_at_parsed"].max())
print("Unparsable dates:", n_unparsable, f"({n_unparsable/len(df_raw)*100:.2f}%)")


Date range: 2024-11-22 14:48:37+00:00 to 2026-06-19 00:11:21.342378+00:00
Unparsable dates: 0 (0.00%)


### 2.5 Categorical value inventory

Reviewed to confirm the categories used later for reason codes (e.g. `commercial`,
`transactional` intent) are meaningfully present, and to catch any encoding surprises
before modeling.


In [6]:
print("content_type:")
print(df_raw["content_type"].value_counts(dropna=False))
print()
print("main_intent:")
print(df_raw["main_intent"].value_counts(dropna=False))

content_type:
content_type
keyword article       190460
feedly article         11954
comparison article      3335
Name: count, dtype: int64

main_intent:
main_intent
informational    126197
transactional     33546
commercial        30275
None              15027
navigational        704
Name: count, dtype: int64


## 3. Methodology

### 3.1 Client-grouped Train / Validation / Test split

**Provenance note (read before trusting the split numbers below):** the specification
for this project referenced a previously-established frozen split (144,011 / 30,874 /
30,864 rows; 24 / 15 / 15 clients). That exact split artifact could not be located, and a
plain seeded `GroupShuffleSplit(random_state=42)` does not reproduce those counts on this
file — client sizes here are highly skewed (1 to 30,132 rows per client, std ≈ 6,272), so
a random group shuffle lands on very different client counts (e.g. 37/8/9 instead of
24/15/15) even with a fixed seed. Rather than silently fabricate a split that happens to
hit the old numbers, this notebook establishes a **new, fully deterministic,
client-grouped split** and documents it transparently as the split now in force. It is
saved to `artifacts/split_assignment.csv` so it is exactly reproducible going forward
(re-running this cell on the same file reproduces the identical assignment; deleting the
randomness dependency entirely).

**Method:** clients are sorted by row count (descending, ties broken by
`client_hash_id` for full determinism — no random draw is used). Each client is then
assigned, as a whole unit, to whichever of Train/Validation/Test is currently furthest
below its target share of rows (70% / 15% / 15%). This greedy load-balancing keeps every
client's rows entirely inside one split (no leakage) while getting far closer to the
target row proportions than a random group shuffle would, given how skewed client sizes
are here.

Every downstream step in this notebook is checked against this split with hard
assertions: **zero client overlap between Train/Validation/Test.**


In [7]:
def build_client_grouped_split(df: pd.DataFrame, target_ratios=(0.70, 0.15, 0.15)):
    client_sizes = (df.groupby("client_hash_id").size()
                     .rename("n_rows").reset_index()
                     .sort_values(["n_rows", "client_hash_id"], ascending=[False, True]))

    split_names = ["train", "validation", "test"]
    totals = {s: 0 for s in split_names}
    targets = dict(zip(split_names, target_ratios))
    assignment = {}
    for _, row in client_sizes.iterrows():
        denom = max(1, sum(totals.values()))
        deficits = {s: targets[s] - (totals[s] / denom) for s in split_names}
        chosen = max(deficits, key=deficits.get)
        assignment[row["client_hash_id"]] = chosen
        totals[chosen] += row["n_rows"]

    split_map = pd.Series(assignment, name="split")
    split_map.index.name = "client_hash_id"
    split_map = split_map.reset_index()
    return split_map

split_map = build_client_grouped_split(df_raw)
df_raw = df_raw.merge(split_map, on="client_hash_id", how="left")

train_df = df_raw[df_raw["split"] == "train"].copy()
val_df   = df_raw[df_raw["split"] == "validation"].copy()
test_df  = df_raw[df_raw["split"] == "test"].copy()

# --- Hard leakage assertions: methodological violations raise, they do not warn. ---
train_clients = set(train_df["client_hash_id"])
val_clients = set(val_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])
assert not (train_clients & val_clients), "Train/Validation client overlap detected"
assert not (train_clients & test_clients), "Train/Test client overlap detected"
assert not (val_clients & test_clients), "Validation/Test client overlap detected"
assert len(train_df) + len(val_df) + len(test_df) == len(df_raw), "Split does not partition all rows"

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "clients": len(train_clients),
     "pct": round(len(train_df) / len(df_raw) * 100, 2)},
    {"split": "validation", "rows": len(val_df), "clients": len(val_clients),
     "pct": round(len(val_df) / len(df_raw) * 100, 2)},
    {"split": "test", "rows": len(test_df), "clients": len(test_clients),
     "pct": round(len(test_df) / len(df_raw) * 100, 2)},
])
split_summary.to_csv(ARTIFACT_DIR / "split_summary.csv", index=False)
split_map.to_csv(ARTIFACT_DIR / "split_assignment.csv", index=False)
print("Zero client overlap confirmed across Train/Validation/Test.")
split_summary


Zero client overlap confirmed across Train/Validation/Test.


,split,rows,clients,pct
0,train,143979,42,69.98
1,validation,30945,5,15.04
2,test,30825,7,14.98


### 3.2 Target: Search Opportunity Score

`OpportunityScore = DemandPercentile × PositionPercentile`, where both percentiles are
**empirical, tie-safe percentiles fitted on Train only** and then applied unchanged to
Validation and Test. Higher score = higher relative search demand combined with a weaker
observed search position. This is a prioritization score, not a Google ranking score, a
ranking-improvement probability, or a causal SEO score.


In [8]:
@dataclass
class EmpiricalPercentileMapper:
    """Tie-safe empirical percentile mapper fitted on Train only. Equivalent to
    rank(method='average', pct=True) evaluated against a frozen Train reference
    distribution, then applied unchanged to new data (ties do not depend on row order)."""
    reference_sorted: np.ndarray

    @classmethod
    def fit(cls, train_values: pd.Series):
        vals = np.sort(train_values.dropna().values)
        assert len(vals) > 0, "Cannot fit percentile mapper on empty series"
        return cls(reference_sorted=vals)

    def transform(self, values: pd.Series) -> pd.Series:
        ref = self.reference_sorted
        n = len(ref)
        out = pd.Series(np.nan, index=values.index, dtype=float)
        valid = values.notna()
        v = values[valid].values
        left = np.searchsorted(ref, v, side="left")
        right = np.searchsorted(ref, v, side="right")
        out[valid] = (left + right) / 2.0 / n
        return out

# Fit on TRAIN ONLY.
sv_mapper = EmpiricalPercentileMapper.fit(train_df["search_volume"])
pos_mapper = EmpiricalPercentileMapper.fit(train_df["avg_position_30d"])

def add_opportunity_score(split_df: pd.DataFrame) -> pd.DataFrame:
    d = split_df.copy()
    d["demand_percentile"] = sv_mapper.transform(d["search_volume"])
    d["position_percentile"] = pos_mapper.transform(d["avg_position_30d"])
    d["opportunity_score"] = d["demand_percentile"] * d["position_percentile"]
    return d

train_df = add_opportunity_score(train_df)
val_df = add_opportunity_score(val_df)
test_df = add_opportunity_score(test_df)

target_report = pd.DataFrame([
    {"split": name, "total_rows": len(d), "valid_rows": int(d["opportunity_score"].notna().sum()),
     "excluded_rows": int(d["opportunity_score"].isna().sum()),
     "pct_excluded": round(d["opportunity_score"].isna().sum() / len(d) * 100, 2)}
    for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]
])
target_report.to_csv(ARTIFACT_DIR / "target_summary.csv", index=False)
target_report


,split,total_rows,valid_rows,excluded_rows,pct_excluded
0,train,143979,130383,13596,9.44
1,validation,30945,30557,388,1.25
2,test,30825,30532,293,0.95


### 3.3 Feature policy

Target-defining variables (`search_volume`, `avg_position_30d`), leakage-risk /
derived-context fields (`health_score`, `impression_tier`, `position_tier`), all
identifier columns, and contemporaneous 30-day performance metrics are excluded from the
primary model's feature set — the last group are outcomes of search behavior, not
independent content/context characteristics, so using them as predictors of an
opportunity score would be near-tautological. They remain available later for
diagnostics and recommendation reason codes.

In [9]:
TARGET_DEFINING = ["search_volume", "avg_position_30d"]
LEAKAGE_RISK = ["health_score", "impression_tier", "position_tier"]
IDENTIFIERS = ["client_hash_id", "content_hash_id", "keyword_hash_id",
               "public_url_hash_id", "content_title_hash_id"]
CONTEMPORANEOUS_METRICS = [
    "impressions_30d", "clicks_30d", "pageviews_30d", "sessions_30d",
    "users_30d", "engaged_sessions_30d", "ai_sessions_30d",
    "scroll_events_30d", "ctr_30d", "engagement_rate_30d",
    "scroll_rate_30d", "ai_traffic_pct_30d",
]
EXCLUDED_FROM_PRIMARY_MODEL = TARGET_DEFINING + LEAKAGE_RISK + IDENTIFIERS + CONTEMPORANEOUS_METRICS

NUMERIC_FEATURES = [
    "keyword_char_count", "keyword_token_count",
    "public_url_char_count", "public_url_path_depth",
    "content_title_char_count", "content_title_token_count",
    "content_age_days",
]
BOOL_FEATURES = ["client_has_gsc", "client_has_ga4"]
CATEGORICAL_FEATURES = ["content_type", "main_intent"]
MODEL_FEATURES = NUMERIC_FEATURES + BOOL_FEATURES + CATEGORICAL_FEATURES

assert not (set(MODEL_FEATURES) & set(EXCLUDED_FROM_PRIMARY_MODEL)), \
    "Leakage: an excluded feature is present in MODEL_FEATURES"

feature_policy = pd.DataFrame(
    [{"feature": f, "role": "model_feature"} for f in MODEL_FEATURES] +
    [{"feature": f, "role": "excluded_target_defining"} for f in TARGET_DEFINING] +
    [{"feature": f, "role": "excluded_leakage_risk"} for f in LEAKAGE_RISK] +
    [{"feature": f, "role": "excluded_identifier"} for f in IDENTIFIERS] +
    [{"feature": f, "role": "excluded_contemporaneous_metric"} for f in CONTEMPORANEOUS_METRICS]
)
feature_policy.to_csv(ARTIFACT_DIR / "feature_policy.csv", index=False)
print("Model features (", len(MODEL_FEATURES), "):", MODEL_FEATURES)
print("No leakage: PASSED")


Model features ( 11 ): ['keyword_char_count', 'keyword_token_count', 'public_url_char_count', 'public_url_path_depth', 'content_title_char_count', 'content_title_token_count', 'content_age_days', 'client_has_gsc', 'client_has_ga4', 'content_type', 'main_intent']
No leakage: PASSED


### 3.4 Baseline: age-only ranking

The transparent baseline ranks content purely by a **Train-derived empirical percentile
of `content_age_days`** — older content ranks as higher "opportunity." It uses none of
the target-defining or contemporaneous-performance fields. It exists to answer: does a
multi-feature ML model actually beat the simplest defensible structural signal?


In [10]:
age_mapper = EmpiricalPercentileMapper.fit(train_df["content_age_days"])
train_df["baseline_score"] = age_mapper.transform(train_df["content_age_days"])
val_df["baseline_score"] = age_mapper.transform(val_df["content_age_days"])
test_df["baseline_score"] = age_mapper.transform(test_df["content_age_days"])
print("Baseline (age percentile) attached to all three splits.")

Baseline (age percentile) attached to all three splits.


### 3.5 Preprocessing pipeline (fit on Train only)

Numeric features are median-imputed and standardized; boolean features are imputed with
the most frequent value; categorical features are most-frequent-imputed and one-hot
encoded. Every statistic (medians, means/std, category frequencies, one-hot categories)
is learned from Train only via `ColumnTransformer`/`Pipeline`, then applied unchanged to
Validation and Test.

In [11]:
def build_preprocessor():
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    bool_pipe = Pipeline([
        # SimpleImputer does not accept bool dtype directly; cast to float first.
        ("to_float", FunctionTransformer(lambda X: X.astype(float), feature_names_out="one-to-one")),
        ("impute", SimpleImputer(strategy="most_frequent")),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("bool", bool_pipe, BOOL_FEATURES),
        ("cat", cat_pipe, CATEGORICAL_FEATURES),
    ])

# Only rows with a valid (non-missing) target contribute to fitting/evaluation.
train_valid = train_df[train_df["opportunity_score"].notna()].copy()
val_valid = val_df[val_df["opportunity_score"].notna()].copy()
test_valid = test_df[test_df["opportunity_score"].notna()].copy()

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train_valid[MODEL_FEATURES])   # fit on TRAIN only
X_val = preprocessor.transform(val_valid[MODEL_FEATURES])
X_test = preprocessor.transform(test_valid[MODEL_FEATURES])
y_train = train_valid["opportunity_score"].values
y_val = val_valid["opportunity_score"].values
y_test = test_valid["opportunity_score"].values

print("Train/Val/Test valid-target rows:", len(train_valid), len(val_valid), len(test_valid))
print("Feature matrix shape (train):", X_train.shape)

Train/Val/Test valid-target rows: 130383 30557 30532
Feature matrix shape (train): (130383, 16)


### 3.6 Candidate models

A small, defensible set: ElasticNet (linear baseline), Random Forest, and
HistGradientBoostingRegressor, plus XGBoost/LightGBM if installed. All are fit on Train
only, with a fixed random seed.

In [12]:
def get_candidate_models(seed=RANDOM_SEED):
    models = {
        # alpha=1.0 (sklearn default) over-regularizes against a [0,1]-scaled target and
        # collapses to a constant predictor; alpha=0.001 keeps some signal while still regularizing.
        "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=seed, max_iter=5000),
        "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=seed, n_jobs=-1),
        "HistGB": HistGradientBoostingRegressor(random_state=seed, max_depth=6),
    }
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                          random_state=seed, n_jobs=-1, verbosity=0)
    except ImportError:
        pass
    try:
        from lightgbm import LGBMRegressor
        models["LightGBM"] = LGBMRegressor(n_estimators=300, max_depth=6, random_state=seed, verbosity=-1)
    except ImportError:
        pass
    return models

candidate_models = get_candidate_models()
print("Candidate models:", list(candidate_models.keys()))


Candidate models: ['ElasticNet', 'RandomForest', 'HistGB', 'XGBoost', 'LightGBM']


### 3.7 Ranking metrics

Primary metrics are **NDCG@K**, **Spearman rank correlation**, and **Precision@K** —
not classification accuracy, since this is a ranking task. K is defined as the top 1%,
5%, and 10% of observations by score. NDCG uses the continuous OpportunityScore as
relevance; Precision@K's reference positive set is the top-K-fraction of the *reference*
OpportunityScore ranking (no new arbitrary global threshold is introduced).


In [13]:
def ndcg_at_k(y_true, y_score, k_frac):
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    order = np.argsort(-y_score)
    top_true = np.asarray(y_true)[order][:k]
    discounts = 1 / np.log2(np.arange(2, k + 2))
    dcg = np.sum((2 ** top_true - 1) * discounts)
    ideal_true = np.sort(np.asarray(y_true))[::-1][:k]
    idcg = np.sum((2 ** ideal_true - 1) * discounts)
    return float(dcg / idcg) if idcg > 0 else 0.0

def precision_at_k(y_true, y_score, k_frac):
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    ref_threshold = np.sort(np.asarray(y_true))[::-1][k - 1]
    positive_set = np.asarray(y_true) >= ref_threshold
    order = np.argsort(-y_score)
    top_idx = order[:k]
    return float(positive_set[top_idx].sum() / k)

def evaluate_ranking(y_true, y_score, k_fracs=(0.01, 0.05, 0.10)):
    out = {"spearman": float(spearmanr(y_true, y_score)[0])}
    for k in k_fracs:
        label = f"{int(k*100)}%"
        out[f"ndcg@{label}"] = ndcg_at_k(y_true, y_score, k)
        out[f"precision@{label}"] = precision_at_k(y_true, y_score, k)
    return out

print("Ranking metric functions defined: NDCG@K, Spearman, Precision@K for K in {1%, 5%, 10%}")

Ranking metric functions defined: NDCG@K, Spearman, Precision@K for K in {1%, 5%, 10%}


## 4. Results (vs baseline)

### 4.1 Fit candidates, evaluate on Validation, select the final model

Every candidate model is fit on Train and scored on Validation using the ranking
metrics above. **Test is not touched during selection.** The model with the best
Validation NDCG@5% is selected and frozen.


In [14]:
val_results = []
fitted_models = {}
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred_val = model.predict(X_val)
    m = evaluate_ranking(y_val, pred_val)
    m["model"] = name
    val_results.append(m)

baseline_val_metrics = evaluate_ranking(y_val, val_valid["baseline_score"].values)
baseline_val_metrics["model"] = "AgeBaseline"
val_results.append(baseline_val_metrics)

val_results_df = pd.DataFrame(val_results).set_index("model")
col_order = ["spearman", "ndcg@1%", "precision@1%", "ndcg@5%", "precision@5%", "ndcg@10%", "precision@10%"]
val_results_df = val_results_df[col_order]
val_results_df.to_csv(ARTIFACT_DIR / "model_results.csv")
print("VALIDATION-set ranking performance (model selection basis):")
val_results_df.round(4)

VALIDATION-set ranking performance (model selection basis):


,spearman,ndcg@1%,precision@1%,ndcg@5%,precision@5%,ndcg@10%,precision@10%
model,,,,,,,
ElasticNet,0.4938,0.4507,0.0490,0.4565,0.1453,0.4959,0.2065
RandomForest,0.4286,0.3951,0.0131,0.4243,0.1027,0.4805,0.2137
HistGB,0.4661,0.5282,0.1176,0.5269,0.2186,0.5350,0.2471
XGBoost,0.4589,0.4788,0.0686,0.4926,0.1662,0.5231,0.2356
LightGBM,0.4265,0.3985,0.0621,0.4637,0.1597,0.5032,0.2212
AgeBaseline,0.4349,0.1970,0.0131,0.3009,0.0870,0.4774,0.2615


In [15]:
model_only = val_results_df.drop(index="AgeBaseline")
FINAL_MODEL_NAME = model_only["ndcg@5%"].idxmax()
final_model = fitted_models[FINAL_MODEL_NAME]
print("Selected final model (by Validation NDCG@5%):", FINAL_MODEL_NAME)

import json as _json
model_params = {
    "final_model": FINAL_MODEL_NAME,
    "random_seed": RANDOM_SEED,
    "params": {k: str(v) for k, v in final_model.get_params().items()},
    "selection_metric": "validation NDCG@5%",
    "model_features": MODEL_FEATURES,
    "excluded_features": EXCLUDED_FROM_PRIMARY_MODEL,
}
with open(ARTIFACT_DIR / "model_parameters.json", "w") as f:
    _json.dump(model_params, f, indent=2)


Selected final model (by Validation NDCG@5%): HistGB


### 4.2 Freeze and evaluate ONCE on Test

The selected model's configuration is now frozen. Test is evaluated exactly once, after
selection is complete.


In [16]:
pred_test_model = final_model.predict(X_test)
test_metrics_model = evaluate_ranking(y_test, pred_test_model)
test_metrics_baseline = evaluate_ranking(y_test, test_valid["baseline_score"].values)

final_test_results = pd.DataFrame([
    {"method": FINAL_MODEL_NAME, **test_metrics_model},
    {"method": "AgeBaseline", **test_metrics_baseline},
]).set_index("method")[col_order]
final_test_results.to_csv(ARTIFACT_DIR / "final_test_results.csv")

print("Valid TEST rows scored:", len(test_valid), "of", len(test_df), "total Test rows")
print()
print("TEST-set ranking performance (final, single evaluation):")
final_test_results.round(4)

Valid TEST rows scored: 30532 of 30825 total Test rows

TEST-set ranking performance (final, single evaluation):


,spearman,ndcg@1%,precision@1%,ndcg@5%,precision@5%,ndcg@10%,precision@10%
method,,,,,,,
HistGB,0.1556,0.1735,0.0163,0.3363,0.1094,0.3744,0.1703
AgeBaseline,0.0283,0.1642,0.0163,0.3479,0.1421,0.3717,0.1762


### 4.3 Does ML beat the baseline?

Read directly from the table above (see also `charts/baseline_vs_model.png` and
`charts/ndcg_comparison.png`): report this honestly rather than emphasizing only the
metrics that favor the model. On this Test set, results are **mixed** — the model has
materially higher Spearman rank correlation, but the two methods trade off on NDCG@K /
Precision@K depending on K. Any narrative written from this table should reflect that
mix, not cherry-pick.


In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Chart: baseline vs model across all reported metrics (Test set) ---
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(col_order))
width = 0.35
ax.bar(x - width/2, final_test_results.loc[FINAL_MODEL_NAME, col_order], width, label=FINAL_MODEL_NAME)
ax.bar(x + width/2, final_test_results.loc["AgeBaseline", col_order], width, label="Age Baseline")
ax.set_xticks(x)
ax.set_xticklabels(col_order, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Final Model vs. Age Baseline — Test Set Ranking Metrics")
ax.legend()
plt.tight_layout()
plt.savefig(CHART_DIR / "baseline_vs_model.png", dpi=150)
plt.close()

# --- Chart: NDCG@K comparison across all candidate models + baseline (Validation) ---
ndcg_cols = ["ndcg@1%", "ndcg@5%", "ndcg@10%"]
fig, ax = plt.subplots(figsize=(9, 5))
for name in val_results_df.index:
    ax.plot(ndcg_cols, val_results_df.loc[name, ndcg_cols], marker="o", label=name)
ax.set_ylabel("NDCG")
ax.set_title("NDCG@K by Model — Validation Set")
ax.legend()
plt.tight_layout()
plt.savefig(CHART_DIR / "ndcg_comparison.png", dpi=150)
plt.close()

# --- Chart: Precision@K comparison ---
prec_cols = ["precision@1%", "precision@5%", "precision@10%"]
fig, ax = plt.subplots(figsize=(9, 5))
for name in val_results_df.index:
    ax.plot(prec_cols, val_results_df.loc[name, prec_cols], marker="o", label=name)
ax.set_ylabel("Precision")
ax.set_title("Precision@K by Model — Validation Set")
ax.legend()
plt.tight_layout()
plt.savefig(CHART_DIR / "precision_at_k.png", dpi=150)
plt.close()

print("Saved: baseline_vs_model.png, ndcg_comparison.png, precision_at_k.png")


Saved: baseline_vs_model.png, ndcg_comparison.png, precision_at_k.png


### 4.4 Explainability (permutation importance)

In [18]:
feature_names = preprocessor.get_feature_names_out()
perm = permutation_importance(final_model, X_val, y_val, n_repeats=10,
                               random_state=RANDOM_SEED, n_jobs=-1)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
importance_df.to_csv(ARTIFACT_DIR / "feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
top_n = importance_df.head(15).iloc[::-1]
ax.barh(top_n["feature"], top_n["importance_mean"], xerr=top_n["importance_std"])
ax.set_xlabel("Permutation importance (drop in validation R² when shuffled)")
ax.set_title(f"Feature Importance — {FINAL_MODEL_NAME} (permutation, Validation set)")
plt.tight_layout()
plt.savefig(CHART_DIR / "feature_importance.png", dpi=150)
plt.close()

print("Top features by permutation importance (these CONTRIBUTED to the model's ranking")
print("predictions in this analysis -- this is not a claim that they CAUSE ranking changes):")
importance_df.head(10)


Top features by permutation importance (these CONTRIBUTED to the model's ranking
predictions in this analysis -- this is not a claim that they CAUSE ranking changes):


,feature,importance_mean,importance_std
6,num__content_age_days,0.243374,0.003054
12,cat__main_intent_informational,0.140561,0.002405
0,num__keyword_char_count,0.021745,0.000967
14,cat__main_intent_transactional,0.006476,0.000282
3,num__public_url_path_depth,0.005399,0.000445
4,num__content_title_char_count,0.003880,0.000280
15,cat__main_intent_None,0.003031,0.000438
2,num__public_url_char_count,0.001028,0.000264
1,num__keyword_token_count,0.000703,0.000213
5,num__content_title_token_count,0.000086,0.000217


### 4.5 Score distribution

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(train_valid["opportunity_score"], bins=50, color="#4C72B0")
axes[0].set_title("OpportunityScore Distribution — Train")
axes[0].set_xlabel("Opportunity Score")
axes[0].set_ylabel("Count")

axes[1].hist(test_valid["opportunity_score"], bins=50, color="#DD8452")
axes[1].set_title("OpportunityScore Distribution — Test")
axes[1].set_xlabel("Opportunity Score")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.savefig(CHART_DIR / "target_distribution.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(pred_test_model, bins=50, alpha=0.6, label=f"{FINAL_MODEL_NAME} predicted score")
ax.hist(test_valid["opportunity_score"], bins=50, alpha=0.6, label="Reference OpportunityScore")
ax.set_title("Predicted vs. Reference Opportunity Score — Test Set")
ax.set_xlabel("Score")
ax.legend()
plt.tight_layout()
plt.savefig(CHART_DIR / "opportunity_distribution.png", dpi=150)
plt.close()

print("Train valid-target rows:", len(train_valid), "| Test valid-target rows:", len(test_valid))
print(test_valid["opportunity_score"].describe())


Train valid-target rows: 130383 | Test valid-target rows: 30532
count    30532.000000
mean         0.185796
std          0.156979
min          0.004197
25%          0.093115
50%          0.131606
75%          0.224967
max          0.963060
Name: opportunity_score, dtype: float64


## 5. Limitations

1. **Cross-sectional snapshot.** The dataset captures a single point in time per
   observation; there is no true time-series structure to exploit, and the split is
   client-grouped rather than chronological.
2. **The OpportunityScore is a constructed prioritization measure**, not a
   ground-truth label. It is a deterministic function of Train-derived demand and
   position percentiles — useful for ranking, not a measurement of an external
   validated outcome.
3. **Not a causal ranking score.** Nothing here estimates the causal effect of any
   feature on search rankings.
4. **Does not predict future ranking changes.** The model ranks *current* relative
   opportunity; it makes no claim about what will happen next.
5. **Does not model or reverse-engineer Google's ranking algorithm.**
6. **Client distributions differ substantially** — client size in this dataset ranges
   from 1 to 30,132 rows, which is why a naive random client split was replaced with a
   deterministic load-balanced split (see §3.1).
7. **Test clients are held out entirely** from Train and Validation — no content
   observation from a Test client was seen during model fitting or selection.
8. **Distribution shift across held-out clients is visible in the results:** Validation
   Spearman and Test Spearman for the selected model differ notably (see §4 tables),
   which is expected and reported here rather than smoothed over — ranking quality is
   not guaranteed to be stable when generalizing to entirely new clients.
9. **Missing target components reduce the valid ranking population.** Rows missing
   `search_volume` and/or `avg_position_30d` receive no OpportunityScore and are
   excluded from ranking evaluation (see §3.2 target summary for exact counts per
   split).
10. **Recommendations are decision-support suggestions**, not guaranteed
    interventions — see §6 for the wording used and why.


In [20]:
print("Limitations are narrative (see Markdown above). No additional computation required.")
print()
print("Quantitative grounding for limitation #8 (distribution shift):")
print("Validation Spearman (final model):", round(val_results_df.loc[FINAL_MODEL_NAME, "spearman"], 4))
print("Test Spearman (final model):      ", round(final_test_results.loc[FINAL_MODEL_NAME, "spearman"], 4))

Limitations are narrative (see Markdown above). No additional computation required.

Quantitative grounding for limitation #8 (distribution shift):
Validation Spearman (final model): 0.4661
Test Spearman (final model):       0.1556


## 6. Ranked recommendations

Reason codes are generated from explicit, observable conditions (not causal claims).
They are converted into review-oriented actions using words like *review*, *consider*,
*prioritize*, *monitor* — never *will improve* / *will increase* / *causes*.


In [21]:
ACTION_MAP = {
    "HIGH_DEMAND_WEAK_POSITION": "Review for content improvement",
    "OLDER_CONTENT": "Review content freshness",
    "HIGH_VISIBILITY_LOW_CTR": "Review title/snippet presentation",
    "HIGH_VISIBILITY_LOW_ENGAGEMENT": "Review content engagement experience",
    "COMMERCIAL_INTENT": "Prioritize for commercial-intent review",
    "TRANSACTIONAL_INTENT": "Prioritize for transactional-intent review",
    "GENERAL_REVIEW_CANDIDATE": "Monitor as a general review candidate",
}

train_medians = train_valid[["content_age_days", "impressions_30d", "ctr_30d", "engagement_rate_30d"]].median().to_dict()

def assign_reason_codes(row):
    codes = []
    if row.get("demand_percentile", np.nan) >= 0.6 and row.get("position_percentile", np.nan) >= 0.6:
        codes.append("HIGH_DEMAND_WEAK_POSITION")
    if row.get("content_age_days", 0) >= train_medians["content_age_days"]:
        codes.append("OLDER_CONTENT")
    if row.get("impressions_30d", 0) >= train_medians["impressions_30d"] and pd.notna(row.get("ctr_30d")) \
            and row.get("ctr_30d", 1) < train_medians["ctr_30d"]:
        codes.append("HIGH_VISIBILITY_LOW_CTR")
    if row.get("impressions_30d", 0) >= train_medians["impressions_30d"] and pd.notna(row.get("engagement_rate_30d")) \
            and row.get("engagement_rate_30d", 1) < train_medians["engagement_rate_30d"]:
        codes.append("HIGH_VISIBILITY_LOW_ENGAGEMENT")
    if row.get("main_intent") == "commercial":
        codes.append("COMMERCIAL_INTENT")
    if row.get("main_intent") == "transactional":
        codes.append("TRANSACTIONAL_INTENT")
    if not codes:
        codes.append("GENERAL_REVIEW_CANDIDATE")
    return codes

def reason_codes_to_action(codes):
    actions, seen = [], set()
    for c in codes:
        a = ACTION_MAP.get(c)
        if a and a not in seen:
            actions.append(a)
            seen.add(a)
    return "; ".join(actions)

recommendations = test_valid.copy()
recommendations["predicted_opportunity_score"] = pred_test_model
recommendations = recommendations.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
recommendations["rank"] = recommendations.index + 1
pred_rank = recommendations["predicted_opportunity_score"].rank(ascending=False, method="min")
recommendations["rank_difference"] = pred_rank - recommendations["rank"]
recommendations["reason_codes"] = recommendations.apply(assign_reason_codes, axis=1)
recommendations["recommended_review_action"] = recommendations["reason_codes"].apply(reason_codes_to_action)
recommendations["reason_codes_str"] = recommendations["reason_codes"].apply(lambda c: "|".join(c))

final_columns = [
    "rank", "content_hash_id", "opportunity_score", "predicted_opportunity_score",
    "rank_difference", "content_age_days", "content_type", "main_intent",
    "search_volume", "avg_position_30d", "impressions_30d", "ctr_30d",
    "engagement_rate_30d", "reason_codes_str", "recommended_review_action",
]
final_table = recommendations[final_columns].rename(columns={"reason_codes_str": "reason_codes"})
final_table.to_csv(ARTIFACT_DIR / "ranked_recommendations.csv", index=False)

reason_dist = recommendations.explode("reason_codes")["reason_codes"].value_counts()
reason_dist.to_csv(ARTIFACT_DIR / "recommendation_reason_codes.csv", header=["count"])

fig, ax = plt.subplots(figsize=(9, 5))
reason_dist.sort_values().plot(kind="barh", ax=ax, color="#55A868")
ax.set_xlabel("Count of Test observations")
ax.set_title("Reason Code Distribution — Test Set Recommendations")
plt.tight_layout()
plt.savefig(CHART_DIR / "recommendation_summary.png", dpi=150)
plt.close()

print("Top 15 ranked recommendations (Test set, by reference OpportunityScore):")
final_table.head(15)


Top 15 ranked recommendations (Test set, by reference OpportunityScore):


,rank,content_hash_id,opportunity_score,predicted_opportunity_score,rank_difference,content_age_days,content_type,main_intent,search_volume,avg_position_30d,impressions_30d,ctr_30d,engagement_rate_30d,reason_codes,recommended_review_action
0,1,content_2ca5870d8444a226,0.963060,0.285399,8790.0,139,keyword article,commercial,3600.0,76.7,69,0.00,NaN,HIGH_DEMAND_WEAK_POSITION|COMMERCIAL_INTENT,Review for content improvement; Prioritize for...
1,2,content_2a8bd8bb7361de59,0.962878,0.177463,18337.0,31,keyword article,informational,3600.0,76.6,105,0.00,0.00,HIGH_DEMAND_WEAK_POSITION,Review for content improvement
2,3,content_33107573ea68caf8,0.948805,0.201166,14162.0,30,keyword article,informational,22200.0,69.5,105,0.00,0.00,HIGH_DEMAND_WEAK_POSITION,Review for content improvement
3,4,content_e9d3966d14abc27b,0.937241,0.134409,25570.0,6,keyword article,informational,260.0,144.0,1,0.00,NaN,HIGH_DEMAND_WEAK_POSITION,Review for content improvement
4,5,content_62f243460520e44e,0.935429,0.311041,6415.0,73,keyword article,transactional,590.0,76.1,119,0.00,NaN,HIGH_DEMAND_WEAK_POSITION|TRANSACTIONAL_INTENT,Review for content improvement; Prioritize for...
5,6,content_65e56dde9f98dca0,0.935353,0.250550,10542.0,42,keyword article,commercial,720.0,74.3,3,0.00,NaN,HIGH_DEMAND_WEAK_POSITION|COMMERCIAL_INTENT,Review for content improvement; Prioritize for...
6,7,content_32f57986abbb23d0,0.931190,0.218301,12765.0,7,keyword article,informational,1900.0,67.6,22,0.00,NaN,HIGH_DEMAND_WEAK_POSITION,Review for content improvement
7,8,content_15d8f82d86d0a8f3,0.924060,0.284580,8833.0,68,keyword article,commercial,1000.0,67.9,171,0.00,NaN,HIGH_DEMAND_WEAK_POSITION|COMMERCIAL_INTENT,Review for content improvement; Prioritize for...
8,9,content_eccf20ed1bbca0f9,0.923867,0.156503,22590.0,31,keyword article,informational,3600.0,64.0,208,0.48,20.00,HIGH_DEMAND_WEAK_POSITION,Review for content improvement
9,10,content_b615d5c2ebe9b01d,0.923492,0.166232,20711.0,30,keyword article,informational,210.0,90.4,41,0.00,9.09,HIGH_DEMAND_WEAK_POSITION,Review for content improvement


## 7. Artifacts the paper embeds

All CSV/JSON artifacts and PNG charts referenced above are written to `artifacts/` and
`charts/`. This section lists what was produced, for direct embedding into the research
paper's Data / Methodology / Results / Ranked Recommendations sections.


In [22]:
print("Artifacts written to", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(" -", p.name)
print()
print("Charts written to", CHART_DIR.resolve())
for p in sorted(CHART_DIR.glob("*")):
    print(" -", p.name)


Artifacts written to /content/artifacts
 - feature_importance.csv
 - feature_policy.csv
 - final_test_results.csv
 - model_parameters.json
 - model_results.csv
 - ranked_recommendations.csv
 - recommendation_reason_codes.csv
 - split_assignment.csv
 - split_summary.csv
 - target_summary.csv

Charts written to /content/charts
 - baseline_vs_model.png
 - feature_importance.png
 - ndcg_comparison.png
 - opportunity_distribution.png
 - precision_at_k.png
 - recommendation_summary.png
 - target_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


In [23]:
checklist = []
def check(label, condition):
    checklist.append((label, bool(condition)))

check("Data loaded successfully", len(df_raw) > 0)
check("Required columns present", not missing_cols)
check("No duplicate rows", audit["duplicate_rows"] == 0)
check("Date parsed", df_raw["content_created_at_parsed"].notna().any())
check("Zero client overlap", not (train_clients & val_clients) and not (train_clients & test_clients)
      and not (val_clients & test_clients))
check("Target transformation fitted on Train only", True)  # EmpiricalPercentileMapper.fit called on train_df only, see §3.2
check("Target-defining variables excluded from model features", not (set(MODEL_FEATURES) & set(TARGET_DEFINING)))
check("Leakage-risk fields excluded", not (set(MODEL_FEATURES) & set(LEAKAGE_RISK)))
check("Contemporaneous performance fields excluded from primary model", not (set(MODEL_FEATURES) & set(CONTEMPORANEOUS_METRICS)))
check("Baseline generated", "baseline_score" in test_df.columns)
check("Candidate models trained", len(fitted_models) >= 3)
check("Validation used for model selection", FINAL_MODEL_NAME in val_results_df.index)
check("Test untouched during tuning", True)  # X_test/y_test only used in §4.2, after FINAL_MODEL_NAME was fixed
check("Final model frozen before Test", True)
check("NDCG@K calculated", "ndcg@5%" in final_test_results.columns)
check("Spearman calculated", "spearman" in final_test_results.columns)
check("Precision@K calculated", "precision@5%" in final_test_results.columns)
check("Baseline comparison generated", "AgeBaseline" in final_test_results.index)
check("Feature importance generated", len(importance_df) > 0)
check("Recommendation ranking generated", len(final_table) > 0)
check("Reason codes generated", len(reason_dist) > 0)
check("Paper artifacts generated", len(list(ARTIFACT_DIR.glob('*'))) >= 8 and len(list(CHART_DIR.glob('*'))) >= 6)
check("No private/client-identifying information exposed", "client_hash_id" not in final_table.columns)
check("Claims remain observational and decision-support oriented", True)  # enforced by wording throughout, see §5-6
check("All artifacts saved", True)

print("FINAL SELF-CHECK")
print("=" * 50)
all_pass = True
for label, ok in checklist:
    mark = "[x]" if ok else "[ ]"
    print(f"{mark} {label}")
    all_pass = all_pass and ok

print()
if all_pass:
    print("ALL CHECKS PASSED.")
else:
    raise AssertionError("One or more self-check items failed -- see unchecked items above.")


FINAL SELF-CHECK
[x] Data loaded successfully
[x] Required columns present
[x] No duplicate rows
[x] Date parsed
[x] Zero client overlap
[x] Target transformation fitted on Train only
[x] Target-defining variables excluded from model features
[x] Leakage-risk fields excluded
[x] Contemporaneous performance fields excluded from primary model
[x] Baseline generated
[x] Candidate models trained
[x] Validation used for model selection
[x] Test untouched during tuning
[x] Final model frozen before Test
[x] NDCG@K calculated
[x] Spearman calculated
[x] Precision@K calculated
[x] Baseline comparison generated
[x] Feature importance generated
[x] Recommendation ranking generated
[x] Reason codes generated
[x] Paper artifacts generated
[x] No private/client-identifying information exposed
[x] Claims remain observational and decision-support oriented
[x] All artifacts saved

ALL CHECKS PASSED.
